# Capa Plata — TFM TUI (Colab)

**Objetivo:** limpiar, normalizar y unificar las tablas de la capa **Bronce**.

Qué se hace aquí:
- **Conversión de coordenadas** UTM ETRS89 30N (EPSG:25830) → lat/lon (WGS84).
- **Esquema común** para las 4 familias: `nombre`, `categoria`, `lat`, `lon`, `distrito`, `barrio`.
- **Cruce locales ↔ terrazas** por `id_local` (añade `tiene_terraza` y superficie).
- Filtrado de filas sin coordenadas válidas.

Los KPIs agregados (densidad por barrio, accesibilidad, etc.) se calcularán en la capa **Oro**.

## 1. Setup — montaje, rutas y dependencias

In [1]:
!pip install -q pyproj

from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
from pyproj import Transformer

# --- Rutas (mismas carpetas que Bronce) ---
BASE   = "/content/drive/MyDrive/Master/TFM TUI 3"
BRONCE = f"{BASE}/Bronce"
PLATA  = f"{BASE}/Plata"
os.makedirs(PLATA, exist_ok=True)

# Transformador UTM ETRS89 30N -> WGS84 (lat/lon). always_xy=True -> entra (x, y), sale (lon, lat)
_tr = Transformer.from_crs("EPSG:25830", "EPSG:4326", always_xy=True)


def utm_a_latlon(x, y):
    """Convierte columnas UTM (x, y) a (lat, lon). Devuelve NaN donde no se pueda."""
    x = pd.to_numeric(x, errors="coerce")
    y = pd.to_numeric(y, errors="coerce")
    lon, lat = _tr.transform(x.values, y.values)
    lon = pd.Series(lon, index=x.index).where(x.notna() & y.notna())
    lat = pd.Series(lat, index=y.index).where(x.notna() & y.notna())
    return lat, lon


# Recuadro de Madrid ciudad (generoso). Lo que caiga fuera son coordenadas corruptas
# (ceros/nulos en origen que pyproj convierte a valores basura, no NaN).
def en_madrid(df):
    ok = df["lat"].between(40.2, 40.7) & df["lon"].between(-4.0, -3.4)
    return df[ok]


def guardar(df, tabla):
    df.to_parquet(f"{PLATA}/{tabla}.parquet", index=False)
    print(f"OK  {tabla}  ->  {len(df)} filas, {df.shape[1]} columnas")

# ------------------------------------------------------------------
# Agrupador de los 443 epigrafes del censo -> macro-categorias (para el mapa)
# Orden importante: gana la PRIMERA regla que casa.
# ------------------------------------------------------------------
import re

_REGLAS_MACRO = [
    ("restaurantes", r"RESTAURANTE|BAR |BAR$|CAFETERIA|CAFE-BAR|TABERNA|CERVECERIA|COCINA|CHOCOLATERIA|HELADERIA|COMIDAS PREPARADAS|CATERING|PIZZERIA|CHURRERIA"),
    ("alimentacion", r"COMERCIO AL POR MENOR.*ALIMENT|FRUTAS|HORTALIZAS|CARNE|PESCAD|PANADERIA|BOLLERIA|PAN Y|PASTELERIA|BEBIDAS|VINO|CHARCUTERIA|HUEVOS|LACTEOS|CONGELADOS|GOLOSINAS|DIETETICOS|HERBORISTERIA|AUTOSERVICIO|SUPERMERCADO|ULTRAMARINOS|OBRADOR"),
    ("hoteles", r"HOTEL|HOSTAL|PENSION|ALOJAMIENTO|APARTAMENTO TURISTICO|VIVIENDAS TUR|CASAS? DE HUESPEDES|ALBERGUE"),
    ("deporte", r"GIMNASIO|DEPORT|FITNESS|PISCINA|POLIDEPORT"),
    ("cultura", r"MUSEO|BIBLIOTECA|LIBRERIA|TEATRO|CINE|GALERIA DE ARTE|SALA DE EXPOSICION|MONUMENTO|CULTURAL|MUSICA| ARTE"),
    ("ocio", r"OCIO|DISCOTECA|SALA DE FIESTA|SALA DE BAILE|JUEGO|APUESTAS|BINGO|CASINO|ATRACCIONES|ESPECTACUL|KARAOKE|BOLERA|BILLAR"),
    ("salud", r"FARMACIA|CLINICA|DENTAL|ODONTOLOG|ESTOMATOLOG|FISIOTERAP|SANITARI|MEDIC|OPTICA|VETERINARI|ORTOPEDIA|PARAFARMACIA|PSICOLOG|PODOLOG|RESIDENCIALES PARA MAYORES|ASISTENCIA EN ESTABLECIMIENTOS"),
    ("educacion", r"ENSEÑANZA|EDUCACION|EDUCATIVOS|ACADEMIA|COLEGIO|GUARDERIA|FORMACION|AUTOESCUELA|IDIOMAS|ESCUELA|INFANTIL Y PRIMARIA|PRIMER CICLO"),
    ("servicios", r"PELUQUERIA|ESTETICA|BELLEZA|DEPILACION|OFICINA|INMOBILIARI|BANC|INTERMEDIACION MONETARIA|SEGUROS|ASESOR|CONTABILIDAD|AUDITORIA|ABOGAD|JURIDIC|GESTORIA|TALLER|REPARACION|LAVANDERIA|TINTORERIA|ARREGLO DE ROPA|FUNERARI|INFORMATICA|PUBLICIDAD|CONSULTORIA|ADMINISTRATIV|LIMPIEZA|SEGURIDAD|FONTANERIA|ELECTRICA|CONSTRUCCION|ACABADO DE EDIFICIOS|CARPINTERIA|EBANISTERIA|TAPICERIA|CERRAJERIA|MENSAJERIA|TATUAJE|TELECOMUNICACION|LOCUTORIO|ARQUITECTURA|INGENIERIA|DISEÑO|PROFESIONAL|IMPRENTA|AGENCIAS DE VIAJES|FOTOGRAFICOS"),
    ("comercio", r"COMERCIO|PRENDAS DE VESTIR|CALZADO|BAZAR|PRECIO UNICO|JOYERIA|PAPELERIA|FLORISTERIA|MUEBLES|ELECTRODOMESTICOS|FERRETERIA|PERFUMERIA|DROGUERIA|JUGUETE|ESTANCO|TABACOS|QUIOSCO|REGALOS|VEHICULOS|RESPUESTOS|REPUESTOS|VENTA"),
]

def macro_categoria(cat):
    """Mapea un epigrafe del censo a su macro-categoria. NaN -> 'sin_dato'."""
    if pd.isna(cat):
        return "sin_dato"
    c = str(cat).upper()
    for nombre, patron in _REGLAS_MACRO:
        if re.search(patron, c):
            return nombre
    return "otros"

# Para POI (categorias ya limpias) -> mismas macro-categorias, para un mapa coherente
_MACRO_POI = {
    "museo": "cultura", "monumento": "cultura",
    "instalacion_deportiva": "deporte", "piscina": "deporte",
    "parque": "ocio", "sala_ocio": "ocio",
    "mercado": "alimentacion", "mercadillo": "comercio",
}

Mounted at /content/drive


## 2. POI — puntos de interés

Trae `LATITUD`/`LONGITUD` en grados, pero con huecos. Estrategia: usar lat/lon
cuando exista; si falta, convertir desde `COORDENADA-X`/`COORDENADA-Y` (UTM).

In [2]:
poi = pd.read_parquet(f"{BRONCE}/Familia_POI.parquet")

# lat/lon directos (los que ya vienen en grados)
lat_dir = pd.to_numeric(poi["LATITUD"],  errors="coerce")
lon_dir = pd.to_numeric(poi["LONGITUD"], errors="coerce")

# lat/lon convertidos desde UTM (para rellenar huecos)
lat_utm, lon_utm = utm_a_latlon(poi["COORDENADA-X"], poi["COORDENADA-Y"])

poi_plata = pd.DataFrame({
    "nombre":    poi["NOMBRE"],
    "categoria": poi["categoria"],
    "lat":       lat_dir.fillna(lat_utm),
    "lon":       lon_dir.fillna(lon_utm),
    "distrito":  poi["DISTRITO"],
    "barrio":    poi["BARRIO"],
})

# Macro-categoria coherente con la de restaurantes (para un unico filtro de mapa)
poi_plata["macro_categoria"] = poi_plata["categoria"].map(_MACRO_POI).fillna("otros")

antes = len(poi_plata)
poi_plata = en_madrid(poi_plata.dropna(subset=["lat", "lon"]))
print(f"POI: {antes} -> {len(poi_plata)} filas con coordenadas validas en Madrid")

guardar(poi_plata, "POI")
display(poi_plata.head())

POI: 1402 -> 1393 filas con coordenadas validas en Madrid
OK  POI  ->  1393 filas, 7 columnas


,nombre,categoria,lat,lon,distrito,barrio,macro_categoria
1,El Capricho de la Alameda Osuna,parque,40.454464,-3.600096,BARAJAS,ALAMEDA DE OSUNA,ocio
2,Jardines Gregorio Ordóñez,parque,40.433462,-3.678595,SALAMANCA,LISTA,ocio
3,Jardines San Francisco el Grande,parque,40.410022,-3.714420,CENTRO,PALACIO,ocio
4,Jardines de Andrés Saborit,parque,40.446133,-3.698970,CHAMBERI,RIOS ROSAS,ocio
5,Jardines de El Buen Retiro,parque,40.419641,-3.687934,RETIRO,LOS JERONIMOS,ocio


## 3. Restaurantes — censo de locales + cruce con terrazas

Solo hay coordenadas UTM (`coordenada_x_local`/`coordenada_y_local`) → convertir.
Se cruza con terrazas por `id_local` para marcar `tiene_terraza` y sumar superficie.

**Se conservan todos los tipos de negocio** (no solo hostelería). Se añade una
columna `macro_categoria` que agrupa los 443 epígrafes del censo en ~11 grupos
manejables para el mapa (restaurantes, alimentacion, comercio, servicios, salud,
deporte, cultura, ocio, educacion, hoteles, otros).

**Se eliminan las filas sin `nombre`** (que además son exactamente las mismas que
no tienen `categoria`) y los **nombres-basura** del censo (`S/R`, `ROTULO NO
INFORMADO`, `MUESTRA`…): no se pueden mostrar ni identificar en el mapa. Se usa
coincidencia exacta para no borrar negocios reales (p. ej. "Rotulos Martinez").

**Nota:** un mismo local con varios epígrafes del mismo grupo se contaría varias
veces; se deja **una fila por `(id_local, macro_categoria)`**. Un local en dos grupos
distintos (p. ej. hotel + restaurante) sí se conserva en ambos.

In [3]:
rest = pd.read_parquet(f"{BRONCE}/Familia_Restaurantes.parquet")
terr = pd.read_parquet(f"{BRONCE}/Familia_Restaurantes_terrazas.parquet")

# Nos quedamos con los locales abiertos (todos los tipos de negocio, no solo hosteleria)
rest = rest[rest["desc_situacion_local"] == "Abierto"]

# --- Agregado de terrazas por local ---
terr["Superficie_ES"] = pd.to_numeric(terr["Superficie_ES"], errors="coerce")
terr_agg = (terr.groupby("id_local")
                .agg(n_terrazas=("id_terraza", "count"),
                     superficie_terraza=("Superficie_ES", "sum"))
                .reset_index())

# --- Coordenadas UTM -> lat/lon ---
lat, lon = utm_a_latlon(rest["coordenada_x_local"], rest["coordenada_y_local"])

rest_plata = pd.DataFrame({
    "id_local":  rest["id_local"],
    "nombre":    rest["rotulo"],
    "categoria": rest["desc_epigrafe"],
    "lat":       lat,
    "lon":       lon,
    "distrito":  rest["desc_distrito_local"],
    "barrio":    rest["desc_barrio_local"],
})

# --- Quitar filas sin nombre (== sin categoria: son las mismas) ---
antes_nulos = len(rest_plata)
rest_plata = rest_plata.dropna(subset=["nombre"])
print(f"Filas sin nombre eliminadas: {antes_nulos - len(rest_plata)}")

# --- Quitar nombres-basura (placeholders del censo, no negocios reales) ---
# Coincidencia EXACTA (upper+strip): asi no borramos negocios reales como
# "Rotulos Martinez" o "Rotulos Molina S.L.".
NOMBRES_BASURA = {
    "S/R", "SR", "S/R.", "S / R", "MUESTRA",
    "ROTULO NO INFORMADO", "RÓTULO NO INFORMADO",
    "ROTULO SIN INFORMAR", "RÓTULO SIN INFORMAR", "ROTULO NO INFORMADA",
    "SIN ROTULO", "SIN RÓTULO", '"SIN ROTULO"', "SINS ROTULO", "DIN RÓTULO", "SON ROTULO",
    "ROTULO", "RÓTULO", "ROTULOS", "RÓTULOS",
    "NO HAY ROTULO", "NO HAY RÓTULO", "NO TIENE ROTULO", "ROTULO NO SOLICITADO",
    # descripciones de cartel (no son nombres de negocio)
    "ROTULO PUBLICITARIO", "RÓTULO PUBLICITARIO",
    "ROTULO PUBLICITARIO OPACO", "RÓTULO PUBLICITARIO OPACO",
    "ROTULO OPACO", "ROTULO IDENTIFICATIVO", "ROTULO LUMINOSO",
    "ROTULO PUBLICITARIO LUMINOSO", "ROTULO Y BANDERIN PUBLICITARIOS",
}
antes_basura = len(rest_plata)
_nom = rest_plata["nombre"].str.strip().str.upper()
rest_plata = rest_plata[~_nom.isin(NOMBRES_BASURA)]
print(f"Nombres-basura eliminados: {antes_basura - len(rest_plata)}")

# --- Macro-categoria para el mapa (conserva 'categoria' fina por debajo) ---
rest_plata["macro_categoria"] = rest_plata["categoria"].apply(macro_categoria)

# --- Quitar multiactividad DENTRO del mismo macro (restaurantes fantasma) ---
# Un mismo local con 2 epigrafes del mismo grupo (p. ej. CARNICERIA + CARNICERIA-SALCHICHERIA)
# se contaria 2 veces. Nos quedamos con 1 fila por (id_local, macro_categoria).
# Un local que este en 2 macros distintos (hotel + restaurante) SI se conserva en ambos.
antes_dup = len(rest_plata)
rest_plata = rest_plata.drop_duplicates(subset=["id_local", "macro_categoria"], keep="first")
print(f"Multiactividad (mismo macro) eliminada: {antes_dup - len(rest_plata)}")

# --- Cruce con terrazas ---
rest_plata = rest_plata.merge(terr_agg, on="id_local", how="left")
rest_plata["n_terrazas"]         = rest_plata["n_terrazas"].fillna(0).astype(int)
rest_plata["superficie_terraza"] = rest_plata["superficie_terraza"].fillna(0.0)
rest_plata["tiene_terraza"]      = rest_plata["n_terrazas"] > 0

antes = len(rest_plata)
rest_plata = en_madrid(rest_plata.dropna(subset=["lat", "lon"]))
print(f"Restaurantes: {antes} -> {len(rest_plata)} filas con coordenadas validas en Madrid")
print(f"  con terraza: {rest_plata['tiene_terraza'].sum()}")
print("\nReparto por macro_categoria:")
print(rest_plata["macro_categoria"].value_counts().to_string())

guardar(rest_plata, "Restaurantes")
display(rest_plata.head())

Filas sin nombre eliminadas: 11028
Nombres-basura eliminados: 16932
Multiactividad (mismo macro) eliminada: 10492
Restaurantes: 121194 -> 98344 filas con coordenadas validas en Madrid
  con terraza: 6433

Reparto por macro_categoria:
macro_categoria
servicios       27972
comercio        16521
restaurantes    15628
alimentacion    10303
otros            9150
salud            7548
deporte          3700
educacion        3099
ocio             1911
hoteles          1626
cultura           886
OK  Restaurantes  ->  98344 filas, 11 columnas


,id_local,nombre,categoria,lat,lon,distrito,barrio,macro_categoria,n_terrazas,superficie_terraza,tiene_terraza
0,10000003,VITACA,BAR RESTAURANTE,40.426558,-3.700788,CENTRO,JUSTICIA,restaurantes,1,6.48,True
1,10000004,ZAATAR & CO,RESTAURANTE,40.428794,-3.707991,CENTRO,UNIVERSIDAD,restaurantes,1,7.02,True
2,10000013,HOTEL MEDIODIA,CAFETERIA,40.408479,-3.692999,CENTRO,EMBAJADORES,restaurantes,0,0.00,False
3,10000052,MUNE,BAR RESTAURANTE,40.424816,-3.696986,CENTRO,JUSTICIA,restaurantes,0,0.00,False
4,10000071,LA DESCUBIERTA,CAFETERIA,40.415187,-3.702627,CENTRO,SOL,restaurantes,0,0.00,False


## 4. Paradas — transporte público

Ya vienen en WGS84 (`stop_lat`/`stop_lon`). Solo normalizamos nombres y tipos.

> **No** se filtran por recuadro de Madrid ciudad: el GTFS del CRTM cubre toda la
> Comunidad, y conservar las paradas regionales es útil para el cálculo de
> accesibilidad en Oro (una parada cercana puede caer justo fuera del límite municipal).

In [4]:
par = pd.read_parquet(f"{BRONCE}/Familia_Paradas.parquet")

# Quitar paradas exactamente duplicadas (aparecen por el unificado metro/cercanias/bus)
antes_par = len(par)
par = par.drop_duplicates()
print(f"Paradas duplicadas eliminadas: {antes_par - len(par)}")

par_plata = pd.DataFrame({
    "nombre":          par["stop_name"],
    "categoria":       par["tipo_transporte"],
    "lat":             pd.to_numeric(par["stop_lat"], errors="coerce"),
    "lon":             pd.to_numeric(par["stop_lon"], errors="coerce"),
    "tipo_transporte": par["tipo_transporte"],
})

antes = len(par_plata)
par_plata = par_plata.dropna(subset=["lat", "lon"])
print(f"Paradas: {antes} -> {len(par_plata)} filas con coordenadas validas")

guardar(par_plata, "Paradas")
display(par_plata.head())

Paradas duplicadas eliminadas: 0
Paradas: 9553 -> 9553 filas con coordenadas validas
OK  Paradas  ->  9553 filas, 5 columnas


,nombre,categoria,lat,lon,tipo_transporte
0,PLAZA DE CASTILLA,Autobuses Interurbanos,40.46690,-3.68917,Autobuses Interurbanos
1,Plaza de Castilla,Autobuses Interurbanos,40.46682,-3.68918,Autobuses Interurbanos
2,Ascensor,Autobuses Interurbanos,40.46555,-3.68877,Autobuses Interurbanos
3,Intercambiador Superficie,Autobuses Interurbanos,40.46728,-3.68915,Autobuses Interurbanos
4,Ascensor,Autobuses Interurbanos,40.46702,-3.68918,Autobuses Interurbanos


## 5. Comprobación rápida

Chequeo de que las coordenadas caen dentro de un rango razonable para Madrid
(lat ~40.3–40.6, lon ~-3.9–-3.5). Sirve para detectar conversiones mal hechas.

In [5]:
for t in ["POI", "Restaurantes", "Paradas"]:
    df = pd.read_parquet(f"{PLATA}/{t}.parquet")
    fuera = ~df["lat"].between(40.2, 40.7) | ~df["lon"].between(-4.0, -3.4)
    print(f"--- {t} ({len(df)} filas) ---")
    print(f"  lat: {df['lat'].min():.4f} .. {df['lat'].max():.4f}")
    print(f"  lon: {df['lon'].min():.4f} .. {df['lon'].max():.4f}")
    print(f"  fuera de rango Madrid: {fuera.sum()} filas")
    print()

--- POI (1393 filas) ---
  lat: 40.3338 .. 40.5824
  lon: -3.8084 .. -3.5575
  fuera de rango Madrid: 0 filas

--- Restaurantes (98344 filas) ---
  lat: 40.3192 .. 40.6085
  lon: -3.8367 .. -3.5339
  fuera de rango Madrid: 0 filas

--- Paradas (9553 filas) ---
  lat: 39.6040 .. 41.1316
  lon: -4.5804 .. -2.8255
  fuera de rango Madrid: 2583 filas

